# FloodWise — End-to-End Emergency Response Decision System

Research-oriented prototype combining structured intake, NLP extraction, validation, multi-agent reasoning, scenario simulation, human approval, mission execution, and field-update reassessment.

In [ ]:
# ============================================================
# FLOODWISE - COMPLETE FASTAPI BACKEND
# ============================================================

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from datetime import datetime
import os
import json
import threading
import time
import uvicorn

# ------------------------------------------------------------
# 1. CREATE FASTAPI APP
# ------------------------------------------------------------

app = FastAPI(title="FloodWise Backend")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ------------------------------------------------------------
# 2. STORAGE
# ------------------------------------------------------------

BASE_DIR = os.getenv("FLOODWISE_DATA_DIR", "../data/sample")
SUBMISSIONS_DIR = os.path.join(BASE_DIR, "submissions")

os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

print("📁 Storage:", BASE_DIR)


# ------------------------------------------------------------
# 3. HOME / HEALTH CHECK
# ------------------------------------------------------------

@app.get("/")
def home():
    return {
        "status": "online",
        "message": "FloodWise backend is running"
    }


# ------------------------------------------------------------
# 4. RECEIVE FLOODWISE SUBMISSIONS
# ------------------------------------------------------------

@app.post("/api/intake")
async def receive_data(data: dict):

    role = str(data.get("role", "unknown")).lower()

    if role not in {"admin", "user"}:
        role = "unknown"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

    filename = os.path.join(
        SUBMISSIONS_DIR,
        f"{role}_{timestamp}.json"
    )

    latest_file = os.path.join(
        BASE_DIR,
        f"{role}_latest.json"
    )

    responses = data.get("responses", {})

    # --------------------------------------------------------
    # PRINT SUBMISSION
    # --------------------------------------------------------

    print("\n" + "=" * 90, flush=True)
    print("🚨 NEW FLOODWISE SUBMISSION", flush=True)
    print("=" * 90, flush=True)

    print(f"👤 PORTAL   : {role.upper()}", flush=True)
    print(f"🕐 RECEIVED : {timestamp}", flush=True)

    print("\n📋 COLLECTED PORTAL DATA", flush=True)
    print("-" * 90, flush=True)

    for question_id, item in responses.items():

        question = item.get(
            "question",
            question_id
        )

        value = item.get(
            "value",
            "Not answered"
        )

        print(
            f"[{question_id}] {question}",
            flush=True
        )

        print(
            f"    ➜ {value}",
            flush=True
        )

    # --------------------------------------------------------
    # SAVE COMPLETE SUBMISSION
    # --------------------------------------------------------

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=4,
            ensure_ascii=False
        )

    # --------------------------------------------------------
    # SAVE LATEST SUBMISSION
    # --------------------------------------------------------

    with open(
        latest_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=4,
            ensure_ascii=False
        )

    print("\n💾 SAVED", flush=True)
    print(
        f"   This submission : {filename}",
        flush=True
    )

    print(
        f"   Latest {role}    : {latest_file}",
        flush=True
    )

    print("\n✅ DATA RECEIVED SUCCESSFULLY", flush=True)
    print("=" * 90, flush=True)

    return {
        "status": "success",
        "message": "FloodWise data received successfully",
        "role": role,
        "timestamp": timestamp,
        "received_fields": len(responses)
    }


# ============================================================
# 5. START UVICORN
# ============================================================

def start_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


server_thread = threading.Thread(
    target=start_server,
    daemon=True
)

server_thread.start()

time.sleep(3)

print("\n🔥 FLOODWISE FASTAPI SERVER STARTED")
print("🌐 Local URL: http://127.0.0.1:8000")

## 1. Backend and Scenario Intake

In [ ]:
import json, os

BASE_DIR = os.getenv("FLOODWISE_DATA_DIR", "../data/sample")
SUBMISSIONS_DIR = os.path.join(BASE_DIR, "submissions")

def load_json(path):
    if not os.path.exists(path):
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)

admin_data = load_json(f"{BASE_DIR}/admin_latest.json")

users = {}

if os.path.exists(SUBMISSIONS_DIR):
    for file in os.listdir(SUBMISSIONS_DIR):
        if file.startswith("user_") and file.endswith(".json"):
            path = os.path.join(SUBMISSIONS_DIR, file)
            data = load_json(path)
            user_id = file.replace(".json", "")
            users[user_id] = data

print("ADMIN:", "Loaded" if admin_data else "Not found")
print("USERS:", len(users))

for user_id in users:
    print(" ", user_id)

In [ ]:
from gliner import GLiNER

nuner = GLiNER.from_pretrained("numind/NuNER_Zero")

print("✅ NuNER Zero loaded successfully")

## 2. NLP Extraction and Validation

In [ ]:
import json, os

BASE_DIR = os.getenv("FLOODWISE_DATA_DIR", "../data/sample")
SUBMISSIONS_DIR = os.path.join(BASE_DIR, "submissions")

def load_json(path):
    if not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

admin_data = load_json(f"{BASE_DIR}/admin_latest.json")

users = {}

for file in os.listdir(SUBMISSIONS_DIR):
    if file.startswith("user_") and file.endswith(".json"):
        user_id = file[:-5]
        users[user_id] = load_json(os.path.join(SUBMISSIONS_DIR, file))

print("ADMIN:", "Loaded" if admin_data else "Not found")
print("USERS:", len(users))

for user_id in users:
    print(" ", user_id)

In [ ]:
from gliner import GLiNER

nuner = GLiNER.from_pretrained("numind/NuNER_Zero")

entities = {}

for user_id, data in users.items():
    note = data.get("responses", {}).get("custom_notes", {}).get("value", "")

    if note and str(note).strip().lower() not in ["none", "null", "not answered"]:
        entities[user_id] = nuner.predict_entities(
            note,
            [
                "entity",
                "quantity",
                "action",
                "condition",
                "location",
                "time"
            ],
            threshold=0.3
        )
    else:
        entities[user_id] = []

print("\n🔎 EXTRACTED ENTITIES FROM ALL USERS")
print("=" * 70)

for user_id, user_entities in entities.items():
    print(f"\n👤 {user_id}")

    for entity in user_entities:
        print(
            f"Text: {entity['text']} | "
            f"Label: {entity['label']} | "
            f"Score: {entity['score']:.3f}"
        )

In [ ]:
import json, os, re, spacy

nlp = spacy.load("en_core_web_sm")

def load(path):
    if not os.path.exists(path):
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def structured(data):
    if not data or not data.get("responses"):
        return "INVALID", []

    out = []

    for qid, x in data["responses"].items():
        v = x.get("value")
        s = str(v).strip().lower()

        if s in ["", "none", "null", "not answered", "n/a", "na"]:
            continue

        ok = True

        if re.fullmatch(r"-?\d+(\.\d+)?", s):
            ok = float(s) >= 0

        out.append((qid, v, "VALID" if ok else "INVALID"))

    return ("INVALID" if any(x[2] == "INVALID" for x in out) else "VALID"), out

def unstructured(text, user_entities):
    if not text or not text.strip():
        return "VALID", [], []

    valid, review = [], []

    for e in user_entities:
        t = e["text"]
        label = e["label"]
        score = e["score"]

        span = text[e["start"]:e["end"]] == t if "start" in e and "end" in e else t.lower() in text.lower()

        doc = nlp(t)
        nums = [x for x in doc if x.like_num]

        consistent = not (label == "quantity" and not nums) and not (
            label != "quantity" and nums and len(nums) == len(doc)
        )

        if not span:
            review.append((t, "not found in original note"))
        elif not consistent:
            review.append((t, "label mismatch"))
        elif score < 0.45:
            review.append((t, "low confidence"))
        else:
            valid.append((t, label, round(score, 3)))

    return ("NEEDS_REVIEW" if review else "VALID"), valid, review


admin = load("/content/floodwise/admin_latest.json")

admin_status, admin_output = structured(admin)

user_results = {}

for user_id, data in users.items():

    status, output = structured(data)

    note = data.get("responses", {}).get("custom_notes", {}).get("value", "")
    user_entities = entities.get(user_id, [])

    note_status, valid_entities, review = unstructured(
        note,
        user_entities
    )

    user_results[user_id] = {
        "structured_status": status,
        "structured": output,
        "note_status": note_status,
        "entities": valid_entities,
        "review": review
    }


all_reviews = []

for user_id, result in user_results.items():
    all_reviews.extend(result["review"])

invalid_structured = admin_status == "INVALID"

for user_id, result in user_results.items():
    if result["structured_status"] == "INVALID":
        invalid_structured = True

if invalid_structured:
    overall_status = "INVALID"
elif all_reviews:
    overall_status = "NEEDS_REVIEW"
else:
    overall_status = "VALID"


print("\n" + "=" * 70)
print("             FLOODWISE INPUT VALIDATION")
print("=" * 70)

print(f"\nOVERALL: {overall_status}")
print(f"DECISION READY: {overall_status == 'VALID'}")

print("\n🏢 ADMIN:", admin_status)

for q, v, s in admin_output:
    print(f"  ✓ {q}: {v} [{s}]")

print("\n👥 USERS")

for user_id, result in user_results.items():

    print(f"\n  👤 {user_id}")
    print(f"  STRUCTURED: {result['structured_status']}")

    for q, v, s in result["structured"]:
        print(f"    ✓ {q}: {v} [{s}]")

    print(f"  CUSTOM NOTE: {result['note_status']}")

    for t, l, s in result["entities"]:
        print(f"    ✓ {t} | {l} | {s}")

    for t, r in result["review"]:
        print(f"    ⚠ {t} | {r}")

print("\n" + "=" * 70)
print(
    "FINAL:",
    "✓ INPUTS VALID"
    if overall_status == "VALID"
    else "⚠ INPUTS REQUIRE REVIEW"
    if overall_status == "NEEDS_REVIEW"
    else "✗ INPUTS INVALID"
)
print("=" * 70)

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

relations = {}

for user_id, data in users.items():
    note = data.get("responses", {}).get("custom_notes", {}).get("value", "")
    relations[user_id] = []

    if not note or str(note).strip().lower() in ["none", "null", "not answered"]:
        continue

    doc = nlp(note)

    for token in doc:
        if token.dep_ in ["dobj", "pobj", "nsubj", "attr"]:
            relations[user_id].append({
                "subject": token.head.text,
                "object": token.text
            })

print("\n🔗 RELATIONSHIPS FROM ALL USERS")
print("=" * 70)

for user_id, user_relations in relations.items():
    print(f"\n👤 {user_id}")

    for r in user_relations:
        print(f"  {r['subject']} → {r['object']}")

In [ ]:
scenario = {
    "scenario_id": "SCN001",
    "admin": {},
    "users": {},
    "custom_notes": [],
    "relations": relations
}

for qid, item in admin_data.get("responses", {}).items():
    value = item.get("value")
    if value is not None and str(value).strip().lower() not in ["", "none", "null", "not answered"]:
        scenario["admin"][qid] = value

for user_id, data in users.items():
    scenario["users"][user_id] = {}

    for qid, item in data.get("responses", {}).items():
        value = item.get("value")

        if value is not None and str(value).strip().lower() not in ["", "none", "null", "not answered"]:
            scenario["users"][user_id][qid] = value

    note = data.get("responses", {}).get("custom_notes", {}).get("value", "")

    if note and str(note).strip().lower() not in ["", "none", "null", "not answered"]:
        scenario["custom_notes"].append({
            "user_id": user_id,
            "text": note,
            "entities": entities.get(user_id, [])
        })

print("\n" + "="*70)
print("             UNIFIED SCENARIO STATE")
print("="*70)

print("\nSCENARIO:", scenario["scenario_id"])

print("\n🏢 ADMIN INPUT")
for k, v in scenario["admin"].items():
    print(f"  {k}: {v}")

print("\n👥 USER / CITIZEN INPUT")

for user_id, data in scenario["users"].items():
    print(f"\n  👤 {user_id}")

    for k, v in data.items():
        print(f"    {k}: {v}")

print("\n📝 CUSTOM NOTES")

for note in scenario["custom_notes"]:
    print(f"\n  👤 {note['user_id']}")
    print(f"    {note['text']}")

    print("    EXTRACTED INFORMATION:")
    for e in note["entities"]:
        print(f"      {e['text']} → {e['label']}")

print("\n🔗 RELATIONSHIPS")

for user_id, user_relations in scenario["relations"].items():
    if user_relations:
        print(f"\n  👤 {user_id}")

        for r in user_relations:
            print(f"    {r['subject']} → {r['object']}")

print("\n" + "="*70)

In [ ]:
def validate_scenario(scenario):
    issues = []
    confirmed = []

    empty = ["", "none", "null", "not answered", "n/a", "na"]

    if scenario.get("admin"):
        confirmed.append("Admin data available")
    else:
        issues.append("Admin data missing")

    valid_users = 0

    for user_id, data in scenario.get("users", {}).items():
        usable = {
            k: v for k, v in data.items()
            if v is not None and str(v).strip().lower() not in empty
        }

        if usable:
            valid_users += 1

    if valid_users:
        confirmed.append(f"{valid_users} user(s) available")
    else:
        issues.append("No usable user data")

    if scenario.get("custom_notes"):
        confirmed.append(
            f"{len(scenario['custom_notes'])} custom note(s) available"
        )

    if scenario.get("relations"):
        confirmed.append(
            f"{len(scenario['relations'])} relationships extracted"
        )

    status = "INVALID" if issues else "VALID"

    return {
        "status": status,
        "confirmed": confirmed,
        "issues": issues
    }


validation = validate_scenario(scenario)

print("\n" + "="*60)
print("             SCENARIO VALIDATION")
print("="*60)

print(f"\nSTATUS: {validation['status']}")

print("\n✓ CONFIRMED")
for x in validation["confirmed"]:
    print("  ✓", x)

print("\n✗ ISSUES")
for x in validation["issues"]:
    print("  ✗", x)

print("\n" + "="*60)

## 3. AI Agent Architecture

In [ ]:
import os
from openai import OpenAI

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    raise RuntimeError("Set NVIDIA_API_KEY in your environment before running the AI agents.")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)
MODEL_NAME = os.getenv("FLOODWISE_MODEL", "nvidia/nemotron-3.5-lightning-30b-a3b")
print(f"AI client ready: {MODEL_NAME}")


In [ ]:
AGENT_DESCRIPTIONS = {
    "SITUATION_AGENT": "Analyzes the emergency situation, affected people, risks and uncertainties.",
    "RESOURCE_AGENT": "Analyzes available, requested, missing and uncertain resources.",
    "REPORT_AGENT": "Analyzes citizen reports and identifies conflicts, confirmations and missing information.",
    "PLANNING_AGENT": "Creates candidate response plans using the available situation and resource information.",
    "SIMULATION_ENGINE": "Tests candidate plans under current and what-if scenarios.",
    "DECISION_AGENT": "Evaluates simulation results and produces an AI recommendation.",
    "HUMAN_APPROVAL": "Authorized human reviews, modifies, approves or rejects the recommendation."
}

print("Agents loaded:")
for agent in AGENT_DESCRIPTIONS:
    print("✓", agent)

In [ ]:
shared_state = {
    "scenario": scenario,
    "validation": validation,
    "situation": None,
    "resources": None,
    "reports": None,
    "plans": None,
    "simulation": None,
    "decision": None,
    "human_approval": None,
    "execution": None
}

print("SHARED STATE INITIALIZED")

In [ ]:
import json

def orchestrator_agent(shared_state):

    completed = {
        "SITUATION_AGENT": shared_state.get("situation") is not None,
        "RESOURCE_AGENT": shared_state.get("resources") is not None,
        "REPORT_AGENT": shared_state.get("reports") is not None,
        "PLANNING_AGENT": shared_state.get("plans") is not None,
        "SIMULATION_ENGINE": shared_state.get("simulation") is not None,
        "DECISION_AGENT": shared_state.get("decision") is not None,
        "HUMAN_APPROVAL": shared_state.get("human_approval") is not None
    }

    state_version = shared_state.get("state_version") or 0
    analysis_version = shared_state.get("analysis_version") or 0

    state_changed = state_version > analysis_version

    if state_changed:

        system_prompt = """
You are the FloodWise Orchestrator.

A field update has changed the Decision Twin.

Your task is to determine which specialist capability should reassess
the changed information.

Available capabilities:

SITUATION_AGENT:
Reassesses emergency conditions, risks, affected people and situation changes.

RESOURCE_AGENT:
Reassesses resource availability, requests, shortages and resource issues.

REPORT_AGENT:
Reassesses citizen reports, conflicts and newly reported information.

PLANNING_AGENT:
Creates new candidate response plans using the updated state.

SIMULATION_ENGINE:
Tests candidate plans against the updated state.

DECISION_AGENT:
Evaluates simulation results and recommends the strongest plan.

Choose the FIRST capability that needs new analysis because of the
field update.

If the field update concerns:
- emergency conditions or road/situation changes → SITUATION_AGENT
- resource shortages, resource requirements or resource availability → RESOURCE_AGENT
- new citizen/report information → REPORT_AGENT

If specialist analysis is already sufficient and a new plan is needed
→ PLANNING_AGENT

If plans exist but the updated state requires them to be tested again
→ SIMULATION_ENGINE

If simulation results are outdated → DECISION_AGENT

Return ONLY the exact capability name.
"""

        user_prompt = f"""
UPDATED DECISION TWIN:

{json.dumps(
    shared_state.get("scenario", {}).get("decision_twin", {}),
    indent=2,
    default=str
)}

LATEST FIELD UPDATE:

{json.dumps(
    shared_state.get("field_updates", [])[-1],
    indent=2,
    default=str
)}

CURRENT ANALYSIS:

Situation:
{json.dumps(shared_state.get("situation"), indent=2, default=str)}

Resources:
{json.dumps(shared_state.get("resources"), indent=2, default=str)}

Reports:
{json.dumps(shared_state.get("reports"), indent=2, default=str)}

Plans:
{json.dumps(shared_state.get("plans"), indent=2, default=str)}

Determine which capability must reassess the changed state.

Return ONLY one capability name.
"""

    else:

        system_prompt = """
You are the FloodWise Orchestrator.

Coordinate the FloodWise specialist capabilities.

Choose the most useful incomplete capability.

Priority:

1. SITUATION_AGENT
2. RESOURCE_AGENT
3. REPORT_AGENT
4. PLANNING_AGENT
5. SIMULATION_ENGINE
6. DECISION_AGENT
7. HUMAN_APPROVAL

Return ONLY the exact capability name.
"""

        user_prompt = f"""
Current completion state:

{json.dumps(completed, indent=2)}

Choose the next capability.
"""

    completion = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "text"},
        temperature=0,
        max_tokens=30,
        stream=False,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    response = completion.choices[0].message.content.strip().upper()

    valid_agents = [
        "SITUATION_AGENT",
        "RESOURCE_AGENT",
        "REPORT_AGENT",
        "PLANNING_AGENT",
        "SIMULATION_ENGINE",
        "DECISION_AGENT",
        "HUMAN_APPROVAL"
    ]

    for agent in valid_agents:
        if agent in response:
            return agent

    return None

In [ ]:
def execute_next_agent(shared_state):

    next_agent = orchestrator_agent(shared_state)

    if next_agent == "SITUATION_AGENT":
        result = situation_agent(shared_state)

    elif next_agent == "RESOURCE_AGENT":
        result = resource_agent(shared_state)

    elif next_agent == "REPORT_AGENT":
        result = report_agent(shared_state)

    elif next_agent == "PLANNING_AGENT":
        result = planning_agent(shared_state)

    elif next_agent == "SIMULATION_ENGINE":
        result = simulation_engine(shared_state)

    elif next_agent == "DECISION_AGENT":
        result = decision_agent(shared_state)

    elif next_agent == "HUMAN_APPROVAL":
        result = human_approval(shared_state)

    else:
        raise ValueError(f"Unknown agent selected: {next_agent}")

    return {
        "selected_agent": next_agent,
        "result": result
    }

In [ ]:
import json

def situation_agent(shared_state):

    system_prompt = """
You are the Situation Agent of FloodWise.

Analyze the validated FloodWise scenario.

Identify only:
- emergency situation
- affected area
- event
- severity
- risks
- affected/citizen reports
- resource mentions
- uncertainties

Do not recommend actions.
Do not create plans.
Do not allocate resources.
Do not invent information.

Return ONLY valid JSON:

{
  "situation": "",
  "affected_area": "",
  "event": "",
  "severity": "",
  "risks": [],
  "citizen_reports": [],
  "resource_mentions": [],
  "uncertainties": []
}
"""

    user_prompt = f"""
FloodWise Scenario:

{json.dumps(shared_state["scenario"], indent=2, default=str)}

Return only the required JSON.
"""

    completion = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0,
        max_tokens=1000,
        stream=False,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    response = completion.choices[0].message.content.strip()
    response = response.replace("```json", "").replace("```", "").strip()

    try:
        result = json.loads(response)
    except:
        start = response.find("{")
        end = response.rfind("}")
        result = json.loads(response[start:end+1])

    shared_state["situation"] = result
    return result

In [ ]:
import json

def resource_agent(shared_state):

    scenario = shared_state["scenario"]

    admin_questions = """
1. Primary Sector / Location Code
2. Primary Trigger Event Type
3. Estimated Total Population Affected
4. Estimated Count Requiring Immediate Evacuation
5. Current Regional Severity Level
6. Inundation Water-Level Trend
7. Major Road Network Status
8. Critical Infrastructure Risk Status
9. Operational Rescue Boats Available
10. Evacuation Buses / Trucks Available
11. Emergency Ambulances Deployed
12. Active Rescue Teams on Ground
13. Relief Camp / Shelter Capacity Status
14. Primary Optimization Strategy Objective
15. Maximum Target Evacuation Window
16. Demographic Priority Group
17. Custom Scenario Prompts & Resource Overrides
"""

    user_questions = """
1. Specific Location / Landmark / Landmark Tag
2. Current Situation Status
3. Number of People Stranded With You
4. Medical / Special Assistance Requirement
5. Immediate Threat Level
6. Local Water Level Progression
7. Ability to Evacuate On Foot
8. Primary Immediate Need
9. Upload Photo Evidence of Situation
10. Additional Emergency Notes or Contact Info
"""

    system_prompt = """
You are the Resource Agent of FloodWise.

Analyze the emergency scenario and identify the resource state.

Use the supplied question definitions to understand the meaning of
the submitted fields.

IMPORTANT RULES:

1. Interpret fields according to their question definitions.
2. Extract administrative resource availability when explicitly provided.
3. Extract citizen resource requests from their immediate needs.
4. Preserve user_id for every citizen request.
5. Do not invent quantities.
6. A request is NOT an allocation.
7. A request is NOT automatically a shortage.
8. Only report a shortage when demand clearly exceeds known availability.
9. If a resource is requested but its availability is not provided,
   put it in unknown_availability.
10. If a requested resource has known availability but requested quantity
    is unknown, do not declare a shortage.
11. Only put resources in allocated_resources when explicitly allocated
    or deployed.
12. Do not convert phrases such as "need food" into numeric quantities.
13. Do not include NuNER entity objects.
14. Do not create plans.
15. Do not recommend actions.
16. Do not invent facts.
17. Do not leave unknown_availability empty when a requested resource has
    no known availability.
18. Return ONLY valid JSON.
19. Do not use markdown.
20. Do not add explanations outside JSON.

Definitions:

available_resources:
Resources whose availability and quantity are explicitly stated.

requested_resources:
Resources explicitly requested by citizens.

allocated_resources:
Resources explicitly stated as allocated or deployed.

resource_shortages:
Only confirmed shortages where demand exceeds known availability.

unknown_availability:
Requested or needed resources for which availability is not provided.

resource_uncertainties:
Known resources or requests where sufficiency, operational capability,
capacity, or deployment conditions cannot be determined.

Return exactly:

{
  "available_resources": [],
  "requested_resources": [],
  "allocated_resources": [],
  "resource_shortages": [],
  "unknown_availability": [],
  "resource_uncertainties": []
}

For available_resources use:

{
  "resource_type": "",
  "quantity": 0,
  "source": ""
}

For requested_resources use:

{
  "resource_type": "",
  "user_id": ""
}

For unknown_availability use:

{
  "resource_type": "",
  "reason": ""
}

For resource_uncertainties use:

{
  "resource_type": "",
  "uncertainty": ""
}

Keep the output concise.
"""

    user_prompt = f"""
ADMIN QUESTION DEFINITIONS:
{admin_questions}

CITIZEN QUESTION DEFINITIONS:
{user_questions}

FLOODWISE SCENARIO:
{json.dumps(scenario, indent=2, default=str)}

Extract the resource state from this scenario.

Return ONLY valid JSON.
"""

    completion = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=1500,
        stream=False,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    response = completion.choices[0].message.content or ""

    print("\nRAW RESOURCE RESPONSE:")
    print(repr(response))

    try:
        result = json.loads(response)

        required_keys = [
            "available_resources",
            "requested_resources",
            "allocated_resources",
            "resource_shortages",
            "unknown_availability",
            "resource_uncertainties"
        ]

        for key in required_keys:
            if key not in result:
                result[key] = []

    except json.JSONDecodeError as e:
        print("\nRESOURCE JSON ERROR:")
        print(e)

        result = {
            "available_resources": [],
            "requested_resources": [],
            "allocated_resources": [],
            "resource_shortages": [],
            "unknown_availability": [],
            "resource_uncertainties": [
                "Resource Agent returned invalid JSON."
            ]
        }

    shared_state["resources"] = result

    return result

In [ ]:
import json

def report_agent(shared_state):

    system_prompt = """
You are the Report Agent of FloodWise.

Analyze all citizen/user reports.

Identify:
- important reports
- confirmed information
- conflicts
- missing information
- uncertainties

Preserve report source IDs.

Be concise.
Do not repeat information.
Do not create plans.
Do not allocate resources.
Do not make decisions.
Do not invent facts.

Return ONLY valid JSON.
Do not use markdown.
Do not include explanations.

Use exactly this structure:

{
  "important_reports": [
    {
      "source": "",
      "content": "",
      "priority": "high"
    }
  ],
  "confirmed_information": [],
  "conflicts": [],
  "missing_information": [],
  "report_uncertainties": []
}

Limits:
important_reports: maximum 5
confirmed_information: maximum 8
conflicts: maximum 5
missing_information: maximum 5
report_uncertainties: maximum 5

Keep every item short.
"""

    user_prompt = f"""
FloodWise Scenario:

{json.dumps(shared_state["scenario"], indent=2, default=str)}

Situation Agent:

{json.dumps(shared_state["situation"], indent=2, default=str)}

Resource Agent:

{json.dumps(shared_state["resources"], indent=2, default=str)}

Analyze the citizen reports.
Return ONLY the JSON object.
"""

    completion = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0,
        max_tokens=2500,
        stream=False,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    response = completion.choices[0].message.content.strip()

    print("\nRAW MODEL RESPONSE:\n")
    print(response)

    response = response.replace("```json", "").replace("```", "").strip()

    start = response.find("{")
    end = response.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("Report Agent did not return a JSON object.")

    json_text = response[start:end + 1]

    try:
        result = json.loads(json_text)
    except json.JSONDecodeError as e:
        print("\nINVALID JSON FROM MODEL")
        print("\nError:", e)
        print("\nJSON returned by model:\n")
        print(json_text)
        raise

    required_keys = [
        "important_reports",
        "confirmed_information",
        "conflicts",
        "missing_information",
        "report_uncertainties"
    ]

    for key in required_keys:
        if key not in result:
            result[key] = []

    shared_state["reports"] = result

    return result

## 4. Agent Execution Pipeline

In [ ]:
import json

def planning_agent(shared_state):

    system_prompt = """
You are the Planning Agent of FloodWise.

Create candidate emergency response plans using the available scenario,
situation, resource and citizen report information.

Your job is to propose plans, not execute them.

Consider:
- affected people
- severity and risks
- available resources
- requested resources
- resource shortages
- citizen reports
- conflicts and uncertainties
- operational constraints

Never assume unavailable resources are available.
Never invent facts.
Do not allocate resources.
Do not make the final decision.
Do not claim a route is safe unless supported by the available data.

Return ONLY valid JSON.
Do not use markdown.

Use exactly this structure:

{
  "candidate_plans": [
    {
      "plan_id": "PLAN_1",
      "objective": "",
      "priority": "",
      "actions": [],
      "resources_required": [],
      "constraints": [],
      "risks": [],
      "assumptions": []
    }
  ],
  "planning_uncertainties": []
}

Create 2 or 3 candidate plans.
Keep them concise.
"""

    user_prompt = f"""
FloodWise Scenario:

{json.dumps(shared_state["scenario"], indent=2, default=str)}

Situation Agent:

{json.dumps(shared_state["situation"], indent=2, default=str)}

Resource Agent:

{json.dumps(shared_state["resources"], indent=2, default=str)}

Report Agent:

{json.dumps(shared_state["reports"], indent=2, default=str)}

Create candidate response plans.
Return ONLY valid JSON.
"""

    completion = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0,
        max_tokens=1800,
        stream=False,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    response = completion.choices[0].message.content.strip()

    print("\nRAW MODEL RESPONSE:\n")
    print(response)

    response = response.replace("```json", "").replace("```", "").strip()

    start = response.find("{")
    end = response.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("Planning Agent did not return JSON.")

    try:
        result = json.loads(response[start:end + 1])
    except json.JSONDecodeError as e:
        print("\nINVALID JSON:")
        print(response)
        print("\nERROR:", e)
        raise

    if "candidate_plans" not in result:
        result["candidate_plans"] = []

    if "planning_uncertainties" not in result:
        result["planning_uncertainties"] = []

    shared_state["plans"] = result

    return result

In [ ]:
def simulation_engine(shared_state):

    scenario = shared_state["scenario"]
    situation = shared_state["situation"]
    resources = shared_state["resources"]
    plans = shared_state["plans"]

    people = 0

    for user_id, data in scenario.get("users", {}).items():
        for key, value in data.items():
            if "people" in str(key).lower() or "person" in str(key).lower():
                try:
                    people += int(float(value))
                except:
                    pass

    available = resources.get("available_resources", [])
    requested = resources.get("requested_resources", [])
    shortages = resources.get("resource_shortages", [])

    results = []

    for plan in plans.get("candidate_plans", []):

        required = plan.get("resources_required", [])

        resource_count = len(required)
        shortage_count = len(shortages)

        coverage = 100

        if people > 0:
            coverage = max(
                0,
                min(100, 100 - (shortage_count * 15))
            )

        risk = "LOW"

        if len(plan.get("risks", [])) >= 3:
            risk = "HIGH"
        elif len(plan.get("risks", [])) >= 1:
            risk = "MEDIUM"

        results.append({
            "plan_id": plan.get("plan_id"),
            "estimated_coverage_percent": coverage,
            "required_resource_count": resource_count,
            "resource_shortage_count": shortage_count,
            "risk_level": risk,
            "feasible": shortage_count == 0
        })

    result = {
        "scenario_id": scenario.get("scenario_id"),
        "plan_results": results,
        "simulation_type": "scenario_based",
        "assumptions": [
            "Simulation uses currently available scenario information",
            "Resource shortages reduce estimated coverage",
            "Risk is derived from plan constraints and identified risks"
        ]
    }

    shared_state["simulation"] = result

    return result

In [ ]:
import json

def decision_agent(shared_state):

    system_prompt = """
You are the Decision Agent of FloodWise.

Evaluate the candidate plans using the simulation results,
situation, resources and citizen reports.

Your job is to recommend the strongest plan.

Do not execute the plan.
Do not allocate resources.
Do not claim certainty when information is uncertain.
The final decision belongs to an authorized human.

Consider:
- feasibility
- estimated coverage
- resource requirements
- resource shortages
- risk
- uncertainties
- operational constraints

Return ONLY valid JSON.
Do not use markdown.
Do not explain your answer outside the JSON.

Use exactly this structure:

{
  "recommended_plan": "",
  "reason": "",
  "comparison": [
    {
      "plan_id": "",
      "score": 0,
      "strengths": [],
      "weaknesses": []
    }
  ],
  "decision_risks": [],
  "confidence": ""
}
"""

    user_prompt = f"""
Evaluate the following FloodWise scenario.

SCENARIO:
{json.dumps(shared_state["scenario"], indent=2, default=str)}

SITUATION:
{json.dumps(shared_state["situation"], indent=2, default=str)}

RESOURCES:
{json.dumps(shared_state["resources"], indent=2, default=str)}

REPORTS:
{json.dumps(shared_state["reports"], indent=2, default=str)}

PLANS:
{json.dumps(shared_state["plans"], indent=2, default=str)}

SIMULATION:
{json.dumps(shared_state["simulation"], indent=2, default=str)}

Recommend the strongest candidate plan.

Return ONLY one valid JSON object.
"""

    completion = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0,
        max_tokens=1500,
        stream=False,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    response = completion.choices[0].message.content or ""

    print("\nRAW DECISION RESPONSE:")
    print(repr(response))

    response = response.strip()
    response = response.replace("```json", "").replace("```", "").strip()

    start = response.find("{")
    end = response.rfind("}")

    if start == -1 or end == -1:
        print("\n⚠️ Decision Agent did not return valid JSON.")

        result = {
            "recommended_plan": None,
            "reason": "Decision Agent returned an invalid response.",
            "comparison": [],
            "decision_risks": ["Invalid model response."],
            "confidence": "LOW"
        }

        shared_state["decision"] = result
        return result

    try:
        result = json.loads(response[start:end + 1])

    except json.JSONDecodeError as e:
        print("\n⚠️ JSON parsing failed:")
        print(e)

        result = {
            "recommended_plan": None,
            "reason": "Decision Agent returned malformed JSON.",
            "comparison": [],
            "decision_risks": ["Malformed model response."],
            "confidence": "LOW"
        }

    shared_state["decision"] = result

    return result

In [ ]:
def human_approval(shared_state):

    decision = shared_state.get("decision")

    if not decision or not decision.get("recommended_plan"):
        raise ValueError("No valid AI recommendation available.")

    print("\n" + "="*60)
    print("             HUMAN APPROVAL")
    print("="*60)

    print("\nAI RECOMMENDATION")
    print("Plan       :", decision["recommended_plan"])
    print("Reason     :", decision["reason"])
    print("Confidence :", decision["confidence"])

    print("\nPLAN COMPARISON")

    for plan in decision.get("comparison", []):
        print(f"\n{plan['plan_id']}")
        print("Score      :", plan["score"])
        print("Strengths  :", ", ".join(plan["strengths"]) or "-")
        print("Weaknesses :", ", ".join(plan["weaknesses"]) or "-")

    print("\nDECISION RISKS")

    for risk in decision.get("decision_risks", []):
        print(" -", risk)

    print("\n" + "-"*60)
    print("1. APPROVE")
    print("2. MODIFY")
    print("3. REJECT")
    print("-"*60)

    choice = input("\nAdmin choice: ").strip()

    if choice == "1":
        result = {
            "status": "APPROVED",
            "selected_plan": decision["recommended_plan"],
            "ai_recommendation": decision["recommended_plan"],
            "approved_by": "ADMIN"
        }

    elif choice == "2":
        selected_plan = input("Enter plan ID to approve: ").strip()

        result = {
            "status": "MODIFIED",
            "selected_plan": selected_plan,
            "ai_recommendation": decision["recommended_plan"],
            "approved_by": "ADMIN"
        }

    elif choice == "3":
        result = {
            "status": "REJECTED",
            "selected_plan": None,
            "ai_recommendation": decision["recommended_plan"],
            "approved_by": "ADMIN"
        }

    else:
        result = {
            "status": "INVALID",
            "selected_plan": None,
            "ai_recommendation": decision["recommended_plan"],
            "approved_by": "ADMIN"
        }

    shared_state["human_approval"] = result

    return result

In [ ]:
import json

def mission_generator(shared_state):

    approval = shared_state.get("human_approval", {})

    if approval.get("status") not in ["APPROVED", "MODIFIED"]:
        raise ValueError("Mission cannot be generated without human approval.")

    selected_plan = approval.get("selected_plan")

    plans = shared_state["plans"].get("candidate_plans", [])

    plan = next(
        (p for p in plans if p.get("plan_id") == selected_plan),
        None
    )

    if plan is None:
        raise ValueError(f"Selected plan {selected_plan} not found.")

    mission = {
        "mission_id": "MIS_001",
        "plan_id": selected_plan,
        "status": "READY",
        "objective": plan.get("objective"),
        "priority": plan.get("priority"),
        "tasks": plan.get("actions", []),
        "required_resources": plan.get("resources_required", []),
        "constraints": plan.get("constraints", []),
        "risks": plan.get("risks", []),
        "assumptions": plan.get("assumptions", [])
    }

    shared_state["mission"] = mission

    return mission

In [ ]:
def update_mission(shared_state, status, note=""):

    mission = shared_state.get("mission")

    if not mission:
        raise ValueError("No mission available.")

    valid_statuses = [
        "ACCEPTED",
        "IN_PROGRESS",
        "COMPLETED"
    ]

    if status not in valid_statuses:
        raise ValueError("Invalid mission status.")

    mission["status"] = status

    update = {
        "mission_id": mission["mission_id"],
        "status": status,
        "note": note
    }

    if "field_updates" not in shared_state:
        shared_state["field_updates"] = []

    shared_state["field_updates"].append(update)

    return update

In [ ]:
#NEW cell after Mission Generator
def field_update(shared_state, status, note="", evacuated_people=None,
                 road_status=None, resource_issue=None):

    mission = shared_state.get("mission")

    if not mission:
        raise ValueError("No active mission found.")

    valid_statuses = [
        "ACCEPTED",
        "IN_PROGRESS",
        "COMPLETED"
    ]

    if status not in valid_statuses:
        raise ValueError("Invalid mission status.")

    update = {
        "mission_id": mission["mission_id"],
        "status": status,
        "note": note,
        "evacuated_people": evacuated_people,
        "road_status": road_status,
        "resource_issue": resource_issue
    }

    if "field_updates" not in shared_state:
        shared_state["field_updates"] = []

    shared_state["field_updates"].append(update)

    return update

## 5. Human Approval and Mission Lifecycle

In [ ]:
#the Decision Twin Update
def update_decision_twin(shared_state):

    updates = shared_state.get("field_updates", [])

    if not updates:
        return shared_state["scenario"]

    latest = updates[-1]

    twin = shared_state["scenario"].setdefault(
        "decision_twin",
        {}
    )

    if latest.get("status"):
        twin["mission_status"] = latest["status"]

    if latest.get("evacuated_people") is not None:
        twin["evacuated_people"] = latest["evacuated_people"]

    if latest.get("road_status"):
        twin["road_status"] = latest["road_status"]

    if latest.get("resource_issue"):
        twin["resource_issue"] = latest["resource_issue"]

    twin["latest_field_update"] = latest

    return twin

In [ ]:
#mark_state_changed
def mark_state_changed(shared_state):

    shared_state["state_version"] = (
        shared_state.get("state_version", 0) + 1
    )

    shared_state["analysis_version"] = None

    return shared_state["state_version"]

In [ ]:
# ============================================================
# FLOODWISE - EXECUTION AGENT
# ============================================================

def execution_agent(shared_state):

    mission = shared_state.get("mission", {})
    decision = shared_state.get("decision", {})
    field_updates = shared_state.get("field_updates", [])

    print("\n" + "=" * 60)
    print("              FLOODWISE EXECUTION AGENT")
    print("=" * 60)

    if not mission:
        print("❌ No mission available.")
        return shared_state

    # --------------------------------------------------------
    # Current mission information
    # --------------------------------------------------------

    mission_id = mission.get("mission_id", "UNKNOWN")
    status = mission.get("status", "READY")

    print(f"Mission ID : {mission_id}")
    print(f"Status     : {status}")

    # --------------------------------------------------------
    # Start mission
    # --------------------------------------------------------

    if status == "READY":

        mission["status"] = "IN_PROGRESS"

        mission["execution"] = {
            "started": True,
            "phase": "EVACUATION",
            "progress": 0,
            "completed": False
        }

        print("\n🚨 Mission started.")
        print("Phase    : EVACUATION")
        print("Progress : 0%")

    # --------------------------------------------------------
    # Existing execution
    # --------------------------------------------------------

    elif status == "IN_PROGRESS":

        execution = mission.get(
            "execution",
            {}
        )

        current_progress = execution.get(
            "progress",
            0
        )

        # Simple demo progression
        current_progress += 25

        if current_progress >= 100:

            current_progress = 100

            mission["status"] = "COMPLETED"

            execution["completed"] = True
            execution["phase"] = "COMPLETED"

            print("\n✅ Mission completed.")

        else:

            execution["completed"] = False

            print("\n🚑 Mission continuing...")

        execution["progress"] = current_progress

        mission["execution"] = execution

        print(
            f"Progress : {current_progress}%"
        )

    # --------------------------------------------------------
    # Completed mission
    # --------------------------------------------------------

    elif status == "COMPLETED":

        print("\n✅ Mission is already completed.")

    # --------------------------------------------------------
    # Invalid state
    # --------------------------------------------------------

    else:

        print(
            f"\n⚠️ Unknown mission status: {status}"
        )

    # --------------------------------------------------------
    # Save state
    # --------------------------------------------------------

    shared_state["mission"] = mission

    return shared_state

In [ ]:
def reassessment_agent(s):
    u=s.get("field_updates",[])[-1]
    road=u.get("road_status","")
    issue=u.get("resource_issue")
    if road=="BLOCKED" or issue:
        status="REPLAN" if road=="BLOCKED" else "MODIFY"
        risk="CRITICAL" if road=="BLOCKED" else "HIGH"
        action="Select alternate evacuation route" if road=="BLOCKED" else "Allocate additional evacuation transport"
    elif road=="PARTIALLY_BLOCKED":
        status="MODIFY"; risk="HIGH"; action="Use alternate route and continue evacuation"
    else:
        status="CONTINUE"; risk="MEDIUM"; action="Continue current mission"
    s["reassessment"]={"reassessment_status":status,"risk_level":risk,"mission_viable":status!="REPLAN","changed_conditions":[u.get("note","")],"resource_concerns":[issue] if issue else [],"operational_impacts":[road] if road else [],"recommended_actions":[action],"replanning_required":status=="REPLAN","reason":action,"confidence":"HIGH"}
    s["mission"]["status"]="IN_PROGRESS"
    return s

field_update(shared_state,"IN_PROGRESS","Water level has increased near the evacuation route. One road is partially blocked. Rescue teams can still reach the affected area.",120,"PARTIALLY_BLOCKED",None)
shared_state=reassessment_agent(shared_state)
print(json.dumps(shared_state["reassessment"],indent=2))